# Marker Genes And Annotation Review

Use this notebook after running `scripts/08_marker_genes.py` to inspect cluster marker genes, marker-based cell type labels, optional enrichment results, and optional RNA velocity output.

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = Path("..").resolve()
MARKERS_DIR = REPO_ROOT / "results" / "markers"
ANNOTATION_DIR = REPO_ROOT / "results" / "annotation"
VELOCITY_DIR = REPO_ROOT / "results" / "velocity"
PLOTS_DIR = REPO_ROOT / "plots" / "annotation"

adata = ad.read_h5ad(ANNOTATION_DIR / "annotated.h5ad")
markers = pd.read_csv(MARKERS_DIR / "cluster_markers.tsv", sep="\t")
annotations = pd.read_csv(ANNOTATION_DIR / "cluster_annotations.tsv", sep="\t")

adata

In [ ]:
annotations.sort_values("leiden")

In [ ]:
display(Image(filename=PLOTS_DIR / "umap_leiden.png"))
display(Image(filename=PLOTS_DIR / "umap_cell_type.png"))

In [ ]:
top_markers = (
    markers.sort_values(["group", "scores"], ascending=[True, False])
    .groupby("group")
    .head(10)
)

top_markers[["group", "names", "scores", "logfoldchanges", "pvals_adj"]]

In [ ]:
top_marker_names = (
    top_markers.groupby("group")["names"]
    .apply(lambda genes: ", ".join(genes.astype(str)))
    .to_frame("top_marker_genes")
)

top_marker_names

In [ ]:
cell_type_counts = adata.obs["cell_type"].value_counts().to_frame("n_cells")
cell_type_counts

In [ ]:
cluster_by_cell_type = pd.crosstab(adata.obs["leiden"], adata.obs["cell_type"])
cluster_by_cell_type

In [ ]:
try:
    import scanpy as sc

    genes_to_plot = top_markers["names"].dropna().astype(str).unique()[:12]
    sc.pl.dotplot(adata, genes_to_plot, groupby="leiden", use_raw=adata.raw is not None)
except ImportError:
    print("Install scanpy to draw dotplots in this notebook.")

In [ ]:
enrichment_path = MARKERS_DIR / "enrichment.tsv"

if enrichment_path.exists():
    enrichment = pd.read_csv(enrichment_path, sep="\t")
    display(enrichment.head(20))
else:
    print("No enrichment.tsv found. Run scripts/08_marker_genes.py --run-enrichment to generate it.")

In [ ]:
velocity_path = VELOCITY_DIR / "velocity.h5ad"

if velocity_path.exists():
    velocity = ad.read_h5ad(velocity_path)
    display(velocity)
else:
    print("No velocity.h5ad found. RNA velocity requires spliced and unspliced layers.")

Review notes:

- Treat automatic annotations as a first pass, not final truth.
- Confirm each cluster using top marker genes and known biology.
- If a cluster label looks wrong, add or edit marker sets and rerun `scripts/08_marker_genes.py --marker-sets <file>`.
- Differential expression here is cluster-vs-rest marker testing from `rank_genes_groups`.
- Enrichment requires `gseapy` and internet access to Enrichr.
- RNA velocity requires spliced/unspliced count layers, which are not present in ordinary 10x gene-count matrices.